# Generate Predictions for Annotation
Runs the fine-tuned Cellpose model on image patches and saves predictions as `.npy` files.
Download the zip and put the contents in the `pred_masks/` directory for model-assisted annotation.

In [ ]:
!pip install -q "opencv-python-headless>=4.9.0.80" cellpose==3.1.1.2

In [ ]:
import numpy as np
import os, glob
from cellpose import core, models

use_GPU = core.use_gpu()
print(f"GPU: {['NO','YES'][use_GPU]}")

## Upload Files
Upload two files:
1. A `.zip` containing `img_*.npy` patches
2. The fine-tuned model file (`finetuned_best`)

In [ ]:
from google.colab import files
uploaded = files.upload()

# Find the zip and model
zip_file  = [f for f in uploaded.keys() if f.endswith('.zip')]
model_file = [f for f in uploaded.keys() if not f.endswith('.zip')]

assert len(zip_file) == 1, f"Upload exactly 1 zip, got {len(zip_file)}"
assert len(model_file) == 1, f"Upload exactly 1 model file, got {len(model_file)}"

zip_file = zip_file[0]
model_file = model_file[0]

print(f"Patches zip: {zip_file}")
print(f"Model file:  {model_file}")

In [ ]:
# Unzip patches
!unzip -qo "{zip_file}" -d /content/patches_in

# Find the actual .npy files (handles nested dirs from zip)
all_npy = sorted(glob.glob('/content/patches_in/**/img_*.npy', recursive=True))
print(f"Found {len(all_npy)} patches")
print(f"Example: {all_npy[0]}")

In [ ]:
# Config
DIAMETER = 17

# Load model
model = models.CellposeModel(gpu=use_GPU, pretrained_model=model_file)
print(f"Model loaded, diameter={DIAMETER}")

# Load all patches
imgs = [np.load(f) for f in all_npy]
print(f"Loaded {len(imgs)} patches")

In [ ]:
# Run predictions
preds, _, _ = model.eval(imgs, diameter=DIAMETER, channels=[0, 0])

# Save
os.makedirs('/content/pred_masks', exist_ok=True)
for f, pred in zip(all_npy, preds):
    name = os.path.basename(f).replace('img_', 'pred_')
    np.save(f'/content/pred_masks/{name}', pred.astype(np.int32))

print(f"Saved {len(preds)} predictions to /content/pred_masks")

In [ ]:
# Preview a few
import matplotlib.pyplot as plt

n = min(6, len(imgs))
idx = np.random.choice(len(imgs), n, replace=False)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))

for col, i in enumerate(idx):
    n_cells = len(np.unique(preds[i])) - 1
    axes[0][col].imshow(imgs[i], cmap='gray')
    axes[0][col].set_title(f'patch {i}', fontsize=9)
    axes[0][col].axis('off')
    axes[1][col].imshow(preds[i] > 0, cmap='gray')
    axes[1][col].set_title(f'{n_cells} cells', fontsize=9)
    axes[1][col].axis('off')

plt.suptitle('Sample predictions — sanity check')
plt.tight_layout()
plt.show()

In [ ]:
# Zip and download
!zip -qj /content/pred_masks_out.zip /content/pred_masks/*.npy
print(f"Zip size: {os.path.getsize('/content/pred_masks_out.zip') / 1e6:.1f} MB")

from google.colab import files
files.download('/content/pred_masks_out.zip')